In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 필요한 패키지 설치
!pip install -q adapter-transformers
!pip install -q --upgrade transformers accelerate
!pip install -q --upgrade transformers
!pip install -q peft==0.6.0
!pip install -q ipywidgets
!pip install -q datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adapters 1.2.0 requires transformers~=4.51.3, but you have transformers 5.5.3 which is incompatible.


# Library

In [1]:
# 데이터 전처리
import pandas as pd
import numpy as np
import os
import json
from typing import List, Dict

from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# pytorch
import torch
import transformers
from transformers import  GenerationConfig,  AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# metric
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Utils

In [2]:
# def generate_aspect_prompts(df: pd.DataFrame) -> List[Dict[str, str]]:
#     """
#     Generate a list of prompts for summarizing each review across predefined aspects.

#     Args:
#         df: DataFrame containing the reviews.

#     Returns:
#         A list of dicts, each with keys "instruction", "input", and "output".
#     """
#     base_instruction = (
#         "You are an assistant summarizing user preferences from user reviews "
#         "in the domain of 'Cellphone and Cellphone Accessories'. "
#         "Your task is to generate natural-sounding, multi-sentence summaries "
#         "that reflect user preferences based on various product aspects.\n\n"
#         "Predefined attribute categories: Features, Qualities, Designs, Durabilities, Brands. "
#         "If you identify other relevant aspects not covered by these categories, you may include them.\n\n"
#     )

#     return [
#         {
#             "instruction": base_instruction,
#             "input": str(review),
#             "output": ""
#         }
#         for review in df['text']
#     ]

In [3]:
def generate_aspect_prompts(df: pd.DataFrame) -> List[Dict[str, str]]:
    """
    Generate a list of prompts for summarizing each review across predefined aspects.

    Args:
        df: DataFrame containing the reviews.

    Returns:
        A list of dicts, each with keys "instruction", "input", and "output".
    """
    base_instruction = (
        "You are an assistant summarizing user preferences from user reviews "
        "in the domain of 'Clothing Shoes and Jewelry'. "
        "Your task is to generate natural-sounding, multi-sentence summaries "
        "that reflect user preferences based on various product aspects.\n\n"
        "Predefined attribute categories: Styles, Colors, Fits, Patterns, Brands. "
        "If you identify other relevant aspects not covered by these categories, you may include them.\n\n"
    )

    return [
        {
            "instruction": base_instruction,
            "input": str(review),
            "output": ""
        }
        for review in df['text']
    ]

In [4]:
# 모델에 해야 할 적업을 추가적으로 설명함.
def generate_prompt(instruction, input=None):
    if input:
         return f"""

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""
    else:
        return f"""

### Instruction:
{instruction}

### Response:
"""

In [5]:
# Chunk 데이터 저장
def save_chunk(data, index, save_dir):
    file_path = os.path.join(save_dir, f"output_chunk_{index:03}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"✅ Saved: {file_path} ({len(data)} samples)")

# Load Data

In [6]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v6_final.pkl')

df = df[:10000]

print(df.shape)
df.head()

(10000, 12)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,LLaMA_Output,LLaVA_Output
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on user reviews, it appears that users a...",The image shows a hand holding a set of earrin...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on user reviews, it appears that users p...",The image shows a close-up of a person's ear w...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on this review, it seems that users pref...","The image shows a fashion accessory, specifica..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on user reviews, it appears that users p...",The image shows a sleeveless tank top with a p...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"Based on the user review, it appears that the ...",The image shows a lime green t-shirt with a gr...


In [7]:
df = df.reset_index(drop=True)

In [8]:
# 데이터셋에서 글자 수 확인
df['text'].apply(lambda row : len(row)).describe().astype(int)

,text
count,10000
mean,537
std,460
min,11
25%,253
50%,414
75%,670
max,9711


In [9]:
# 데이터셋에서 공백기준 단어의 수 확인
df['text'].apply(lambda row : len(row.split(' '))).describe().astype(int)

,text
count,10000
mean,100
std,85
min,1
25%,47
50%,78
75%,127
max,1583


## Load LLaMA-3-8B-Instruct

In [10]:
os.environ['HF_TOKEN']="[REDACTED]" # Colab에서는 UI로 입력할 수 있음.
auth_token = "[REDACTED]"

# 모델과 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", cache_dir='./llama3/', token=auth_token)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", device_map="auto", torch_dtype=torch.float16, cache_dir='./llama3/', token=auth_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

### 토크나이저와 모델에서 패딩 토큰 설정

In [11]:
tokenizer.pad_token_id = tokenizer.eos_token_id # "End of Sequence"를 나타내는 토큰의 ID이며, 시퀀스가 끝났음을 알리는 용도로 사용됨.
model.config.pad_token_id = model.config.eos_token_id

In [12]:
tokenizer.padding_side = "left" # 패딩이 오른쪽에서 시작해서 왼쪽 방향으로 채워집니다.

##### **결과 해석**
<table>
  <thead>
    <tr>
      <th style="text-align:left; padding:4px;">태그 또는 항목</th>
      <th style="text-align:left; padding:4px;">설명</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding:4px;"><code>&lt;|begin_of_text|&gt;</code></td>
      <td style="padding:4px;">전체 텍스트나 대화의 시작을 나타냄</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;system&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">“system” 역할 메시지 표시 (AI 지침/배경 정보 포함)</td>
    </tr>
    <tr>
      <td style="padding:4px;">Cutting Knowledge Date: December 2023<br>Today Date: 26 Jul 2024</td>
      <td style="padding:4px;">대화가 발생한 시간적 맥락 제공</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;user&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">사용자가 보낸 메시지를 표시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>Who is Vincent van Gogh?</code></td>
      <td style="padding:4px;">사용자가 제기한 질문 예시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;assistant&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">AI 응답이 시작됨을 표시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|eot_id|&gt;</code></td>
      <td style="padding:4px;">텍스트나 대화의 끝을 나타냄</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;userStyle&gt;Normal&lt;/userStyle&gt;</code></td>
      <td style="padding:4px;">사용자가 일반 응답을 요청함을 표시</td>
    </tr>
  </tbody>
</table>


## Llama 3을 통해 Text 생성하기

In [13]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

## Prompt 확인

In [14]:
# 프롬프트 생성
base_prompts = generate_aspect_prompts(df)

In [15]:
# 데이터에서 모델에 입력할 변수를 추출
instructions = [item['instruction'] for item in base_prompts]
inputs = [item['input'] for item in base_prompts]

In [16]:
instructions[0]

"You are an assistant summarizing user preferences from user reviews in the domain of 'Clothing Shoes and Jewelry'. Your task is to generate natural-sounding, multi-sentence summaries that reflect user preferences based on various product aspects.\n\nPredefined attribute categories: Styles, Colors, Fits, Patterns, Brands. If you identify other relevant aspects not covered by these categories, you may include them.\n\n"

In [17]:
inputs[0]

'Pretty colored earrings but color of stainless steel could be a bit lighter to make it more silvery looking. But overall nice earrings that are comfortable to wear and affordable.'

# LLaMA-3-8B tokenizer 기준 Input token 길이 확인

In [18]:
# # 토큰 개수 계산 함수 정의
# def count_tokens(text):
#     return len(tokenizer.encode(text, add_special_tokens=False))

# # 컬럼 추가
# df['num_tokens'] = df['text'].progress_apply(count_tokens)

In [19]:
# df['num_tokens'].describe()

In [20]:
# sns.histplot(df['num_tokens'], kde=True)
# plt.title('Distribution of Token Counts')
# plt.xlabel('Token Count')
# plt.ylabel('Frequency')
# plt.show()

# Inference and Save Result

In [21]:
generation_config = model.generation_config

generation_config.do_sample = True
generation_config.temperature = 0.001
generation_config.top_p = 1.0
generation_config.top_k = 40
generation_config.num_beams = 1

generation_config.max_new_tokens = 256
generation_config.max_length = None   # ⭐ 핵심

generation_config.return_dict_in_generate = True
generation_config.output_scores = False
generation_config.pad_token_id = tokenizer.pad_token_id
generation_config.use_cache = True

In [ ]:
# 프롬프트 생성
prompts = [
    generate_prompt(instruction=str(instruction), input=str(input))
    for instruction, input in zip(instructions, inputs)
]

# 저장 관련 설정
SAVE_INTERVAL = 1000
save_dir = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_8/"
os.makedirs(save_dir, exist_ok=True)
all_outputs = [None] * len(prompts)
chunk_index = 0

# 배치 단위 생성 및 후처리
BATCH_SIZE = 4
filled_indices = []

for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
    batch_prompt = prompts[i:i + BATCH_SIZE]

    # 토크나이징
    prompt_inputs = tokenizer(
        batch_prompt,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # 모델 응답 생성
    with torch.no_grad():
        generation_output = model.generate(
          **prompt_inputs,
          generation_config=generation_config
        )
    s = generation_output.sequences
    output = tokenizer.batch_decode(s, skip_special_tokens=True)
    output = [_.split('Response:\n')[-1] for _ in output]

    # 결과 저장
    for j, result in enumerate(output):
        global_idx = i + j
        all_outputs[global_idx] = result
        filled_indices.append(global_idx)

    # SAVE_INTERVAL마다 청크 저장
    if len(filled_indices) >= SAVE_INTERVAL:
        chunk_data = []
        saved_indices = []
        count = 0
        for idx in filled_indices:
            if all_outputs[idx] is not None:
                chunk_data.append(all_outputs[idx])
                saved_indices.append(idx)
                count += 1
                if count == SAVE_INTERVAL:
                    break

        save_chunk(chunk_data, chunk_index, save_dir)
        for idx in saved_indices:
            all_outputs[idx] = None
        filled_indices = [idx for idx in filled_indices if idx not in saved_indices]
        chunk_index += 1

# 남은 결과 저장
valid_outputs = [val for val in all_outputs if val is not None]
if valid_outputs:
    save_chunk(valid_outputs, chunk_index, save_dir)

 10%|█         | 250/2500 [35:10<4:39:23,  7.45s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_8/output_chunk_000.json (1000 samples)


 20%|██        | 500/2500 [1:08:53<3:14:13,  5.83s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_8/output_chunk_001.json (1000 samples)


 30%|███       | 750/2500 [1:43:02<3:34:51,  7.37s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_8/output_chunk_002.json (1000 samples)


... (10 outputs left out)

 90%|█████████ | 2250/2500 [5:09:14<27:19,  6.56s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_8/output_chunk_008.json (1000 samples)


100%|██████████| 2500/2500 [5:41:50<00:00,  8.20s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_8/output_chunk_009.json (1000 samples)


# Concat Dataset and Chunk

In [ ]:
# import json

# # 파일 경로
# input_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_1/output_chunk_184.json"
# output_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_1/output_chunk_184_cleaned.json"

# # 1. 파일 읽기
# with open(input_path, "r", encoding="utf-8") as f:
#     data = json.load(f)

# # 2. "[INVALID]" 항목 제거
# cleaned_data = [entry for entry in data if entry.strip() != "[INVALID]"]

# # 3. 다시 저장
# with open(output_path, "w", encoding="utf-8") as f:
#     json.dump(cleaned_data, f, ensure_ascii=False, indent=4)

# print(f"✅ 저장 완료: {output_path} (원래 {len(data)}개 → {len(cleaned_data)}개)")

In [ ]:
# 디렉토리 경로 설정
load_dir = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Fashion_7/"

# 파일 이름 정렬
file_names = sorted([
    f for f in os.listdir(load_dir) if f.startswith("output_chunk_") and f.endswith(".json")
])

# 파일 읽기 및 DataFrame으로 로드
df_list = []
for file_name in file_names:
    file_path = os.path.join(load_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        text_df = pd.DataFrame(data)
        df_list.append(text_df)

# axis=0 방향으로 합치기 (행 방향)
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

# 결과 확인 (옵션)
print(f"총 샘플 수: {len(combined_df)}")
print(combined_df.head())

총 샘플 수: 6718
                                                   0
0  Based on the user reviews, it appears that use...
1  Based on this review, it seems that the user p...
2  Based on this review, it seems that users appr...
3  Based on the user review, it appears that the ...
4  Based on the user's review, here is a summary ...


In [ ]:
combined_df.columns = ['LLaMA_Output']

df = pd.concat([df, combined_df], axis=1)

In [ ]:
pd.set_option('display.max_colwidth', None)

df[['title_y', 'text', 'LLaMA_Output']].tail(20)

,title_y,text,LLaMA_Output
6698,Casio Men's G-Shock G9000-1 Sport Watch,"Llevo unas cuantas semanas usándolo y puedo decir que es mi relog favorito! A pesar de ser G-Sock tien un tamaño discreto, además sus funciones básicas son las que más se podrían usar. Algo que me gusta mucho, es que bo debo preocuparme si se moja!!!!","Based on the user review, here is a summary of their preferences:\n\nThe user is extremely satisfied with their G-Sock watch, which they have been using for a few weeks. They appreciate its compact size, which makes it easy to wear. Additionally, they value the basic functions of the watch, which they find most useful. Furthermore, they are relieved that they don't have to worry about the watch getting wet, which is a significant advantage for them. Overall, the user is very pleased with their G-Sock watch and considers it their favorite."
6699,ACECHA Long Pearl Drop Dangle Earrings for Women 16K Gold Plated Earrings Jewelry for Wedding,"Amazing quality earrings! It comes in a box with a cute flower on it - very nice touch that makes this product gift-able as well. The earrings themselves are high quality, almost like they were luxury or expensive items. I think these earrings would be perfect for an evening out in elegant wear. I'd highly recommend this product!","Based on this review, it appears that the user highly values the quality of the earrings, considering them to be almost luxury-like. They also appreciate the thoughtful packaging, which makes the product gift-worthy. Additionally, the user suggests that the earrings would be suitable for an evening out in elegant wear, indicating a preference for formal or dressy occasions. Overall, the user's preferences seem to focus on high-quality, elegant, and gift-worthy products."
6700,"Flower Earrings, White Tulip Flower Dangle Earrings with S925 Sterling Silver Piercing, Hypoallergy Earrings For Women and Girls","I'm blown away by the high quality of these gold flower earrings. It looks incredibly high-end and is suitable for both evening, elegant wear as well as more casual everyday wear. With regard to the design, the petals are clearly well-made, and the jewels in the middle are shiny and beautiful. I love the gold trimming as well because it gives a nice finality to the earrings. With regard to actual wear, the earrings are not too heavy, so I'm certain they can last a couple hours or a whole evening without weighing down your ears or hurting. The earring bar itself is also not too short like I've found with other earrings, so it doesn't pinch your ear lobe. Overall, beautiful purchase! Highly recommend to buyer.","Based on this review, it appears that the user prefers:\n\n* High-quality products with a luxurious look and feel, as they mention the earrings look ""incredibly high-end"".\n* Versatile designs that can be worn for both formal and casual occasions.\n* Attention to detail in the design, such as well-made petals and shiny jewels.\n* A touch of elegance, as they appreciate the gold trimming.\n* Comfortable wear, as they mention the earrings are not too heavy and the earring bar is not too short.\n* Brands that deliver on quality, as they highly recommend the purchase.\n\nIn terms of specific styles, colors, fits, patterns, and brands, the user seems to prefer:\n\n* Elegant and sophisticated styles, possibly with a touch of glamour.\n* Gold or gold-trimmed designs, which suggests a preference for metallic colors.\n* Comfortable fits, as they mention the earrings are not too heavy or pinchy.\n* Delicate patterns, such as the flower design, which adds a touch of whimsy.\n* High-end or luxury brands that deliver on quality and style.\n\nOverall, the user's preferences suggest a taste for high-end, elegant, and comfortable jewelry with attention to detail and a touch of sophistication."
6701,"Smilephil Gold Name Necklace for Women Men - Pearl Bunny Bear Letter Beads Initial Pendant Chain Choker, Christmas Necklace","I love this necklace so much!

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_review_b7.pkl')